# 📚 Chapter 4. Model I/O (Ollama) 총정리

이 노트북은 AI 모델(Ollama)에게 **"어떻게 질문(Prompt)을 잘 던지고, 어떻게 설정값을 관리할 것인가?"**를 다루는 핵심 파트입니다.

### 🎯 핵심 요약 (실무 개발 시 언제 꺼내봐야 할까?)

#### 1. 퓨샷 프롬프팅 (Few-Shot Prompting)
AI에게 무작정 질문하는 것이 아니라, **"내가 원하는 대답의 모범 형식(답안지 예시)"**을 여러 개 먼저 보여주고 흉내 내게 하는 기술입니다.
* **`4.1 FewShotPromptTemplate`**: 예시들을 하나의 거대한 줄글(통짜 텍스트)로 묶어서 전달하는 기본 방식.
* **`4.2 FewShotChatMessagePromptTemplate` ⭐(권장)**: 구형 4.1과 달리, 예시들을 카카오톡 대화처럼 `[Human]`, `[AI]` 역할별로 명확히 구분해서 챗 모델에게 전달하는 **최신 표준 방식**. (현업에서 무조건 이걸 써야 합니다)

#### 2. 프롬프트 다루기 고급 팁 (Selectors & Composition)
* **`4.3 Example Selectors`**: 내가 준비한 예제가 10개일 때 모두 넘기면 너무 느려집니다. 토큰 낭비를 막기 위해 **상황에 맞는 예제만 랜덤 혹은 길이 기반으로 1~2개 쏙 골라서** 넘겨주는 똑똑한 선택 기능 장치.
* **`4.4 Serialization & Composition`**:
  * **불러오기(Serialization):** 긴 프롬프트 텍스트를 파이썬 코드 안쪽에 지저분하게 적지 않고, 외부에 별도의 `.json`이나 `.yaml` 설정 파일로 저장한 뒤 깔끔하게 `load_prompt`로 당겨쓰는 방법.
  * **조립(Composition):** [상황 부여 템플릿] + [예시 템플릿] + [진짜 질문 템플릿] 등 여러 조각들을 더하기 기호(`+`)로 레고 블록처럼 손쉽게 합쳐 하나의 거대한 프롬프트를 만드는 방법.

#### 3. 성능 최적화의 보물 (Caching) ⭐(무조건 켜기)
* **`4.5 Caching`**: 전에 물어봤던 똑같은 질문이 또 들어오면? AI가 10초 넘게 바보처럼 헛고생하지 않도록, **미리 저장해 둔 내장 데이터베이스(`cache.db`)에서 0.001초 만에 바로 정답을 꺼내서 벼락처럼 대답**하게 하는 강력한 최적화 기술. 

#### 4. 내 AI 세팅 영구 보존 (Model Serialization)
* **`4.6 Model Serialization`**: 내 프로젝트에 딱 맞춰서 섬세하게 조율해 둔 AI 모델 객체의 상세 셋팅값(`model="llama3.1"`, `temperature=0.1` 등) 껍데기 자체를 `model.json` 파일로 복사해서 영구적으로 하드에 저장해 둡니다. 나중에 켤 때 `.load()` 한 줄이면 모델 설정과 성격까지 그대로 똑같이 복원되는 기능입니다. 

In [ ]:
# 4.1 FewShotPromptTemplate

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.callbacks import StreamingStdOutCallbackHandler

chat = ChatOllama(
    model="llama3.3",
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

# 예제의 형식을 지정하는 방법
example_template = """
    Human: {question}
    AI: {answer}
"""

example_prompt = PromptTemplate.from_template(example_template)
# example_prompt = PromptTemplate.from_template("Human: {question}\nAI: {answer}")

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,                      # 프롬프트 템플릿
    examples=examples,                                  # 프롬프트에 입력될 값의 예시
    suffix="Human: What do you know about {country}?",  # 형식화된 모든 예제 마지막에 붙일 내용. 보통 사용자의 질문
    input_variables=["country"],
)

# prompt.format(country="Germany")

chain = prompt | chat

chain.invoke({"country": "Turkey"})

I know this:
Capital: Ankara
Language: Turkish
Food: Doner Kebab and Baklava
Currency: Turkish Lira (not Euro, as Turkey is not a member of the Eurozone)

AIMessage(content='I know this:\nCapital: Ankara\nLanguage: Turkish\nFood: Doner Kebab and Baklava\nCurrency: Turkish Lira (not Euro, as Turkey is not a member of the Eurozone)', additional_kwargs={}, response_metadata={'model': 'llama3.3', 'created_at': '2026-03-23T01:45:55.830466475Z', 'done': True, 'done_reason': 'stop', 'total_duration': 28399784879, 'load_duration': 7615147968, 'prompt_eval_count': 154, 'prompt_eval_duration': 3715536567, 'eval_count': 44, 'eval_duration': 16984417499, 'logprobs': None, 'model_name': 'llama3.3', 'model_provider': 'ollama'}, id='lc_run--019d185e-5084-73f3-a6f5-258e61600d5c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 154, 'output_tokens': 44, 'total_tokens': 198})

In [ ]:
# 4.2 FewShotChatMessagePromptTemplate

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_core.prompts import ChatPromptTemplate

chat = ChatOllama(
    model="llama3.3",
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

examples = [
    {
        "country": "France",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "country": "Italy",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "country": "Greece",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "What do you know about {country}?"),
    ("ai", "{answer}")
])
# example_prompt = ChatPromptTemplate.from_template("Human: {country}\nAI: {answer}")

prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,                      # 프롬프트 템플릿
    examples=examples,                                  # 프롬프트에 입력될 값의 예시
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert, you give short answers"),
    prompt,
    ("human", "What do you know about {country}?")
])

# prompt.format(country="Germany")

chain = final_prompt | chat

chain.invoke({"country": "Turkey"})

Capital: Ankara
Language: Turkish
Food: Kebabs and Doner
Currency: Lira

AIMessage(content='Capital: Ankara\nLanguage: Turkish\nFood: Kebabs and Doner\nCurrency: Lira', additional_kwargs={}, response_metadata={'model': 'llama3.3', 'created_at': '2026-03-23T02:23:33.126859384Z', 'done': True, 'done_reason': 'stop', 'total_duration': 19743675028, 'load_duration': 7623617154, 'prompt_eval_count': 173, 'prompt_eval_duration': 3772261140, 'eval_count': 22, 'eval_duration': 8304994724, 'logprobs': None, 'model_name': 'llama3.3', 'model_provider': 'ollama'}, id='lc_run--019d1880-e3e5-7e72-ba07-4d6604f57559-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 173, 'output_tokens': 22, 'total_tokens': 195})

In [6]:
# 4.3 LengthBasedExampleSelector

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.example_selectors.base import BaseExampleSelector


chat = ChatOllama(
    model="llama3.3",
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

class RandomExampleSelector(BaseExampleSelector):

    def __init__(self, examples):
        self.examples = examples

    def add_example(self, example):
        self.examples.append(example)

    def select_examples(self, input_variables):
        from random import choice
        return [choice(self.examples)]


# 예제의 형식을 지정하는 방법
#example_template = """
#    Human: {question}
#    AI: {answer}
#"""

# example_prompt = PromptTemplate.from_template(example_template)
example_prompt = PromptTemplate.from_template("Human: {question}\nAI: {answer}")

# example_selector = LengthBasedExampleSelector(
#     examples=examples,
#     example_prompt=example_prompt,
#     max_length=180,
# )
example_selector = RandomExampleSelector(
    examples=examples,
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,                      # 프롬프트 템플릿
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",  # 형식화된 모든 예제 마지막에 붙일 내용. 보통 사용자의 질문
    input_variables=["country"],
)

prompt.format(country="Brazil")

'Human: What do you know about Greece?\nAI: \n        I know this:\n        Capital: Athens\n        Language: Greek\n        Food: Souvlaki and Feta Cheese\n        Currency: Euro\n        \n\nHuman: What do you know about Brazil?'

In [11]:
# 4.4 Serialization and Composition

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_core.prompts import load_prompt


prompt = load_prompt('./prompt.json')
prompt = load_prompt('./prompt.yaml')

# prompt.format(country="Germany")

chat = ChatOllama(
    model="llama3.3",
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

intro = PromptTemplate.from_template(
    """
    You are a role playing assistant
    And you are impersonating a {character}
    """
)

example = PromptTemplate.from_template(
    """
    This is an example of how you talk:
    
    Human: {example_question}
    You: {example_answer}
    """
)

start = PromptTemplate.from_template(
    """
    Start now!
    
    Human: {question}
    You:
    """
)

final = PromptTemplate.from_template(
    """
    {intro}
    
    {example}
    
    {start}
    """
)


full_prompt = intro + "\n\n" + example + "\n\n" + start

# full_prompt.format(
#     character='Pirate',
#     example_question='What is your location?',
#     example_answer='Arrrrg! That is a secret!! Arg Arg!!',
#     question='What is your favorite food?',
# )

chain = full_prompt | chat
chain.invoke({
    "character":'Pirate',
    "example_question":'What is your location?',
    "example_answer":'Arrrrg! That is a secret!! Arg Arg!!',
    "question":'What is your favorite food?',
})

Arrr, me hearty! Yer want ta know what fills me belly, eh? Alright then, matey... Me favorite grub be SEA DOG STEW! A mighty mix o' seafood, veggies, and a bit o' treasure (that be the secret ingredient, savvy?). It warms me bones after a long day o' plunderin' and pillagin' on the high seas! Arg arg!

AIMessage(content="Arrr, me hearty! Yer want ta know what fills me belly, eh? Alright then, matey... Me favorite grub be SEA DOG STEW! A mighty mix o' seafood, veggies, and a bit o' treasure (that be the secret ingredient, savvy?). It warms me bones after a long day o' plunderin' and pillagin' on the high seas! Arg arg!", additional_kwargs={}, response_metadata={'model': 'llama3.3', 'created_at': '2026-03-23T06:50:42.349733465Z', 'done': True, 'done_reason': 'stop', 'total_duration': 44545863431, 'load_duration': 7937230270, 'prompt_eval_count': 79, 'prompt_eval_duration': 3483429503, 'eval_count': 85, 'eval_duration': 32952481636, 'logprobs': None, 'model_name': 'llama3.3', 'model_provider': 'ollama'}, id='lc_run--019d1975-192a-7351-8174-7763796b23d3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output_tokens': 85, 'total_tokens': 164})

In [13]:
# 4.5 Caching

from langchain_ollama import ChatOllama
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_core.globals import set_llm_cache, set_debug
from langchain_core.caches import InMemoryCache
from langchain_community.cache import SQLiteCache

# set_llm_cache(InMemoryCache())
# set_debug(True)
set_llm_cache(SQLiteCache("cache.db"))


chat = ChatOllama(
    model="llama3.3",
    temperature=0.1,
    # streaming=True,
    # callbacks=[
    #     StreamingStdOutCallbackHandler(),
    # ],
)

chat.invoke("How do you make Italian pasta").content

"Italian pasta! There are countless ways to make delicious Italian pasta dishes, but I'll provide a general guide on how to cook pasta and some classic Italian pasta recipes.\n\n**Basic Steps:**\n\n1. **Choose your pasta**: Select the type of pasta you want to use (e.g., spaghetti, linguine, fettuccine, penne, etc.).\n2. **Boil water**: Fill a large pot with salted water (about 4-6 quarts for every pound of pasta). Bring the water to a boil.\n3. **Cook the pasta**: Add the pasta to the boiling water and cook until it's al dente (firm to the bite), which is usually between 8-12 minutes, depending on the type of pasta.\n4. **Drain and serve**: Drain the cooked pasta in a colander and return it to the pot. Add your favorite sauce and toss everything together.\n\n**Classic Italian Pasta Recipes:**\n\n1. **Spaghetti Aglio e Olio** (Spaghetti with Garlic and Olive Oil):\n\t* Cook spaghetti according to package instructions.\n\t* In a pan, heat 3-4 tablespoons of olive oil over medium-low hea

In [14]:
chat.invoke("How do you make Italian pasta").content

"Italian pasta! There are countless ways to make delicious Italian pasta dishes, but I'll provide a general guide on how to cook pasta and some classic Italian pasta recipes.\n\n**Basic Steps:**\n\n1. **Choose your pasta**: Select the type of pasta you want to use (e.g., spaghetti, linguine, fettuccine, penne, etc.).\n2. **Boil water**: Fill a large pot with salted water (about 4-6 quarts for every pound of pasta). Bring the water to a boil.\n3. **Cook the pasta**: Add the pasta to the boiling water and cook until it's al dente (firm to the bite), which is usually between 8-12 minutes, depending on the type of pasta.\n4. **Drain and serve**: Drain the cooked pasta in a colander and return it to the pot. Add your favorite sauce and toss everything together.\n\n**Classic Italian Pasta Recipes:**\n\n1. **Spaghetti Aglio e Olio** (Spaghetti with Garlic and Olive Oil):\n\t* Cook spaghetti according to package instructions.\n\t* In a pan, heat 3-4 tablespoons of olive oil over medium-low hea

In [19]:
# 4.6 Serialization
from langchain_ollama import ChatOllama

chat = ChatOllama(
    model="llama3.1",
    temperature=0.1,
)

a = chat.invoke("What is the recipe for soju") 
b = chat.invoke("What is the recipe for bread")

print(a.content, "\n\n", b.content, "\n")

print("\n소주 레시피 토큰 사용량: \n", a.usage_metadata)
print("\n빵 레시피 토큰 사용량: \n", b.usage_metadata)


Soju! Korea's favorite spirit. While traditional soju recipes are closely guarded secrets, I can provide you with a general outline of how soju is typically made and a simplified recipe to try at home. Please note that this recipe may not produce an exact replica of commercial soju, as the manufacturing process involves specialized equipment and techniques.

**Traditional Soju Ingredients:**

* Grains: rice, barley, wheat, or a combination of these
* Nuruk (Korean fermentation starter): a type of koji (Aspergillus oryzae) that breaks down starches into fermentable sugars
* Water

**Simplified Soju Recipe:**

To make a basic soju at home, you'll need:

Ingredients:

* 2 cups of short-grain rice (such as Japanese mochigome or Korean chapssal)
* 1 cup of water
* 1/4 teaspoon of active dry yeast (e.g., sake yeast or champagne yeast)
* 1/4 teaspoon of koji powder (available at Asian markets or online)

Instructions:

1. **Rice preparation**: Rinse the rice thoroughly and soak it in water fo

In [21]:
# 4.6 Serialization
from langchain_ollama.llms import OllamaLLM
chat = OllamaLLM(
    model="llama3.1",
    temperature=0.1,
)
chat.save("model.json")

In [23]:
import json
from langchain_core.load import load

with open("model.json", "r") as f:
    chat = load(json.load(f))

chat

/tmp/ipykernel_103340/586163479.py:5: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  chat = load(json.load(f))


{'_type': 'ollama-llm'}